# Оценка MDE и мощности A/B-теста

В этом ноутбуке оценивается минимальный эффект, который мог быть обнаружен в реконструированном A/B-тесте посадочной страницы.

Из предтестовых данных был рассчитан baseline bounce rate старой страницы `/home` для сегмента `gsearch / nonbrand`:

- baseline bounce rate: **60.99%**
- control group: **2328 сессий**
- treatment group: **2381 сессия**

Гипотезы теста:

- **H0:** bounce rate новой страницы не ниже bounce rate старой страницы;
- **H1:** bounce rate новой страницы ниже bounce rate старой страницы.

Поскольку исходный MDE эксперимента неизвестен, он оценивается ретроспективно через фактический размер выборки.

Для нескольких возможных значений MDE рассчитывается статистическая мощность теста при:

- `alpha = 0.05`
- целевой мощности `power = 0.80`
- односторонней альтернативе

Цель анализа — определить минимальное снижение bounce rate, которое такой эксперимент мог бы обнаружить с мощностью около 80%.

Важно: полученное значение не доказывает, что именно такой MDE был изначально выбран командой. Оно показывает только порог обнаружимого эффекта при фактическом размере выборки и при заданных параметрах (мощность теста, порог значимости).

In [19]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
import pandas as pd
import numpy as np

baseline = 0.6099
n_control = 2328
n_test = 2381
results = []

analysis = NormalIndPower()

for mde in [0.01, 0.015, 0.02, 0.025, 0.03, 0.035, 0.04]:
    treatment = baseline - mde

    effect_size = proportion_effectsize(
        baseline,
        treatment
    )

    power = analysis.power(
        effect_size=effect_size,
        nobs1=n_control,
        ratio=n_test / n_control,
        alpha=0.05,
        alternative='larger'
)

    results.append({
        'MDE, п.п.': mde * 100,
        'Power, %': power * 100
    })

df = pd.DataFrame(results)

df.style.format({
    'MDE, п.п.': '{:.1f}',
    'Power, %': '{:.4f}'
})

,"MDE, п.п.","Power, %"
0,1.0,17.2830
1,1.5,27.6507
2,2.0,40.3558
3,2.5,54.1538
4,3.0,67.4421
5,3.5,78.7980
6,4.0,87.4139


Сузим диапазон MDE до 3.5–4.0 п.п. и выполним более точный поиск значения, при котором мощность теста приближается к 80%.

In [20]:
results = []

for mde in np.arange(0.035, 0.0375, 0.0005):
    treatment = baseline - mde

    effect_size = proportion_effectsize(
        baseline,
        treatment
    )

    power = analysis.power(
        effect_size=effect_size,
        nobs1=n_control,
        ratio=n_test / n_control,
        alpha=0.05,
        alternative='larger'
)

    results.append({
        'MDE, п.п.': mde * 100,
        'Power, %': power * 100
    })

df = pd.DataFrame(results)

df.style.format({
    'MDE, п.п.': '{:.2f}',
    'Power, %': '{:.4f}'
})

,"MDE, п.п.","Power, %"
0,3.50,78.7980
1,3.55,79.7896
2,3.60,80.7528
3,3.65,81.6872
4,3.70,82.5927


Дополнительно сузим диапазон до 3.55–3.60 п.п. с шагом 0.01 п.п.

In [21]:
results = []

for mde in np.arange(0.0355, 0.036, 0.0001):
    treatment = baseline - mde

    effect_size = proportion_effectsize(
        baseline,
        treatment
    )

    power = analysis.power(
        effect_size=effect_size,
        nobs1=n_control,
        ratio=n_test / n_control,
        alpha=0.05,
        alternative='larger'
)

    results.append({
        'MDE, п.п.': mde * 100,
        'Power, %': power * 100
    })

df = pd.DataFrame(results)

df.style.format({
    'MDE, п.п.': '{:.2f}',
    'Power, %': '{:.4f}'
})

,"MDE, п.п.","Power, %"
0,3.55,79.7896
1,3.56,79.9845
2,3.57,80.1783
3,3.58,80.3709
4,3.59,80.5624
5,3.60,80.7528


При фактическом размере выборки эксперимент способен обнаружить снижение bounce rate примерно на 3.56 п.п. с мощностью около 80% при уровне значимости 5%.

Это значение интерпретируется как ретроспективно оценённый detectable effect, а не как подтверждённый исходный MDE, выбранный командой до запуска эксперимента.